## ⚖️ Comparação: Regressão Linear vs. Regressão Polinomial

A principal diferença entre os dois modelos reside na capacidade de capturar relações não lineares entre as variáveis de entrada e o alvo. Abaixo, um resumo comparativo:

| Característica | Regressão Linear | Regressão Polinomial |
| :--- | :--- | :--- |
| **Premissa** | Assume relação em linha reta | Cria combinações não lineares ($x^2, x^3, \dots$) |
| **Vantagem** | Simplicidade e alta interpretabilidade | Alta capacidade de ajuste a padrões curvos |
| **Risco** | *Underfitting* (subajuste) em dados complexos | *Overfitting* (sobreajuste) se o grau for muito alto |



### 1. Regressão Linear (Grau 1)
* **Conceito:** Busca a melhor linha reta que minimiza o erro.
* **Uso:** Ideal para relações simples onde o aumento em $X$ gera uma variação proporcional e constante em $Y$.

### 2. Regressão Polinomial (Grau > 1)
* **Conceito:** Introduz termos de maior grau para permitir que a curva "acompanhe" o comportamento dos dados.
* **Cuidado:** Modelos com grau muito elevado (ex: grau 10) tendem a "decorar" o ruído dos dados de treino, perdendo a capacidade de generalização em novos dados (testes).

---

## 📊 Diagnóstico: Homocedasticidade vs. Heterocedasticidade

Para que um modelo de regressão seja considerado robusto e confiável, a análise dos resíduos (erros) é obrigatória. Não basta olhar o $R^2$; é preciso verificar o comportamento do erro.



### O que é?
A **Homocedasticidade** ocorre quando a variância dos erros é constante para todos os níveis de previsão.

* **Cenário Ideal (Homocedástico):** Os pontos no gráfico de resíduos estão espalhados de forma aleatória e uniforme em torno da linha horizontal do zero. Isso indica que o modelo erra "da mesma forma" para qualquer faixa de valor.
* **Cenário Crítico (Heterocedástico):** Os resíduos formam um desenho de "funil" ou "leque". Isso indica que a variância do erro aumenta (ou diminui) conforme a previsão cresce, sinalizando uma falha estrutural no modelo.

### Como avaliar?
1. **Gráfico de Dispersão de Resíduos (*Residual Plot*):** A forma mais rápida de identificar visualmente padrões não aleatórios.
2. **Testes Estatísticos:** O uso de testes formais, como o de **Breusch-Pagan**, ajuda a validar matematicamente se a variância dos resíduos é constante. 

> **Dica:** Se detectada heterocedasticidade, considere aplicar uma transformação logarítmica na variável alvo (`log(y)`) ou utilizar um modelo que lide melhor com a variância dos dados.

## 📊 Análise de Normalidade dos Resíduos

Após o treinamento dos modelos de regressão, é fundamental verificar se os **resíduos seguem uma distribuição normal**, pois essa é uma das principais premissas da regressão linear clássica.

Foram utilizados os seguintes testes estatísticos:

- **Shapiro-Wilk**
- **Lilliefors**
- **Kolmogorov-Smirnov (KS)**

---

## 🎯 Qual é o valor ideal?

Todos esses testes seguem a mesma lógica:

- **Hipótese nula (H₀):** Os resíduos seguem uma distribuição normal  
- **Hipótese alternativa (H₁):** Os resíduos NÃO seguem uma distribuição normal  

O principal valor analisado é o **p-value**.

---

## ✅ Regra de decisão (considerando α = 0.05)

| p-value | Interpretação |
|----------|---------------|
| **p > 0.05** | Não rejeitamos H₀ → Os resíduos podem ser considerados normais ✅ |
| **p ≤ 0.05** | Rejeitamos H₀ → Os resíduos não seguem distribuição normal ❌ |

---

## 📌 Valores Ideais

Para que o modelo atenda à premissa de normalidade dos resíduos, o ideal é que:

- O **p-value seja maior que 0.05**
- Preferencialmente valores como **0.10, 0.20 ou maiores**, indicando maior evidência de normalidade

Quanto maior o p-value (acima de 0.05), maior a evidência de que os resíduos se aproximam de uma distribuição normal.


In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from scipy.stats import shapiro, kstest, norm
from statsmodels.stats.diagnostic import lilliefors


In [19]:

# 1. Preparação dos Dados
df = pd.read_csv('./datasets/sales_data.csv')
colunas_numericas = ['tempo_de_experiencia', 'numero_de_vendas', 'fator_sazonal']
X = df[colunas_numericas]
y = df['receita_em_reais']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

# 2. Padronização
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- MODELO 1: REGRESSÃO LINEAR ---
lin_model = LinearRegression()
lin_model.fit(X_train_scaled, y_train)
y_pred_lin = lin_model.predict(X_test_scaled)

# --- MODELO 2: REGRESSÃO POLINOMIAL (Grau 2) ---
poly_feat = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly_feat.fit_transform(X_train_scaled)
X_test_poly = poly_feat.transform(X_test_scaled)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)

# --- MÉTRICAS DE DESEMPENHO ---
print("📊 MÉTRICAS DE DESEMPENHO")
print("-" * 50)

r2_lin = r2_score(y_test, y_pred_lin)
rmse_lin = root_mean_squared_error(y_test, y_pred_lin)

r2_poly = r2_score(y_test, y_pred_poly)
rmse_poly = root_mean_squared_error(y_test, y_pred_poly)

print(f"LINEAR     -> R²: {r2_lin:.4f} | RMSE: {rmse_lin:.2f}")
print(f"POLI (G2)  -> R²: {r2_poly:.4f} | RMSE: {rmse_poly:.2f}")

# --- ANÁLISE DE RESÍDUOS (NORMALIDADE) ---
print("\n🔍 TESTES DE NORMALIDADE NOS RESÍDUOS")
print("-" * 50)

# Resíduos Linear
res_lin = y_test - y_pred_lin

# Resíduos Polinomial
res_poly = y_test - y_pred_poly

def testar_normalidade(residuos, nome_modelo):
    print(f"\n📌 {nome_modelo}")

    # Shapiro-Wilk
    stat_shapiro, p_shapiro = shapiro(residuos)
    print(f"Shapiro-Wilk        -> p-value = {p_shapiro:.4f}")

    # Lilliefors
    stat_lilliefors, p_lilliefors = lilliefors(residuos)
    print(f"Lilliefors          -> p-value = {p_lilliefors:.4f}")

    # Kolmogorov-Smirnov (padronizado)
    residuos_std = (residuos - np.mean(residuos)) / np.std(residuos)
    stat_ks, p_ks = kstest(residuos_std, 'norm')
    print(f"Kolmogorov-Smirnov  -> p-value = {p_ks:.4f}")

    # Interpretação rápida
    if p_shapiro > 0.05:
        print("➡ Resíduos parecem seguir distribuição normal (α=0.05)")
    else:
        print("➡ Resíduos NÃO seguem distribuição normal (α=0.05)")

# Comparação
testar_normalidade(res_lin, "Regressão Linear")
testar_normalidade(res_poly, "Regressão Polinomial (Grau 2)")


📊 MÉTRICAS DE DESEMPENHO
--------------------------------------------------
LINEAR     -> R²: -0.0769 | RMSE: 2381.10
POLI (G2)  -> R²: 0.0033 | RMSE: 2290.76

🔍 TESTES DE NORMALIDADE NOS RESÍDUOS
--------------------------------------------------

📌 Regressão Linear
Shapiro-Wilk        -> p-value = 0.2794
Lilliefors          -> p-value = 0.2788
Kolmogorov-Smirnov  -> p-value = 0.6648
➡ Resíduos parecem seguir distribuição normal (α=0.05)

📌 Regressão Polinomial (Grau 2)
Shapiro-Wilk        -> p-value = 0.3726
Lilliefors          -> p-value = 0.4867
Kolmogorov-Smirnov  -> p-value = 0.8136
➡ Resíduos parecem seguir distribuição normal (α=0.05)
